# Control Specificity Check

This notebook applies the current best locked imaging biomarker, the SRM Global Linear Composite, to healthy/control participants. Controls are not used for training, preprocessing estimation, tuning, or model selection. The notebook code is intentionally self-contained for review, while reusing established `src/` model and metric helpers.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repository root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import DEFAULT_CONFIG, set_global_seeds
from src.data.audit import modelling_pair_count_table
from src.data.qc import standardize_train_test
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.intervals import interval_effect_summary, pooled_adjacent_pair_effect_summary
from src.models.srm_global import SRMGlobalLinear

set_global_seeds(DEFAULT_CONFIG.random_state)
RANDOM_SEED = DEFAULT_CONFIG.random_state
N_BOOT = 500

WIDE_PATH = REPO_ROOT / "data" / "processed" / "trackfa_merged_wide.csv"
PAIRS_PATH = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
SRM_LOG_PATH = REPO_ROOT / "results" / "srm_composite_optimization_log.csv"
MODEL_PERFORMANCE_PATH = REPO_ROOT / "results" / "model_performance_summary.csv"

wide_df = pd.read_csv(WIDE_PATH)
pairs_df = pd.read_csv(PAIRS_PATH)

modelling_counts = modelling_pair_count_table(pairs_df, expected={"N12": 108, "N23": 99, "N13": 90, "N123": 90})
print("Canonical modelling-cohort pair counts")
display(modelling_counts)
pair_long_df = trackfa_pairs_to_long(pairs_df)
feature_groups = infer_trackfa_feature_groups(pairs_df)

LOCKED_MODEL = {
    "model": "SRM Global Linear Composite",
    "selection_method": "none",
    "ridge": 1e-6,
    "covariance_shrinkage": 0.45,
    "z_clip": None,
    "training_data": "FRDA annual pairs from trackfa_pairs_drop3poms.csv",
}

imaging_cols = [
    c for c in feature_groups.all_neuroimaging
    if c in pair_long_df.columns and all(f"{c}_v{visit}" in wide_df.columns for visit in (1, 2, 3))
]
print(f"Loaded wide table: {wide_df.shape[0]} participants")
print(f"Loaded FRDA annual pair table: {pairs_df.shape[0]} annual intervals")
print(f"Locked feature panel available in both FRDA pair table and wide table: {len(imaging_cols)} MRI features")
display(pd.DataFrame([LOCKED_MODEL]))


Canonical modelling-cohort pair counts


,count,interval,n,definition
0,N12,V1->V2,108,subjects with a V1V2 annual pair row
1,N23,V2->V3,99,subjects with a V2V3 annual pair row
2,N13,V1->V3,90,subjects with both V1V2 and V2V3 annual pair rows
3,N123,"V1,V2,V3",90,subjects represented across all three visits v...


Loaded wide table: 269 participants
Loaded FRDA annual pair table: 207 annual intervals
Locked feature panel available in both FRDA pair table and wide table: 146 MRI features


,model,selection_method,ridge,covariance_shrinkage,z_clip,training_data
0,SRM Global Linear Composite,none,0.000001,0.45,None,FRDA annual pairs from trackfa_pairs_drop3poms...


## Prepare FRDA Training Rows And Control Visit Rows

`study_group == 0` is treated as FRDA/patient data and `study_group == 1` is treated as healthy/control data, matching the existing project guardrail. The model is fitted on FRDA annual pairs only. Control rows are scored after applying the FRDA-fitted standardisation parameters.


In [2]:
def wide_to_visit_long(wide: pd.DataFrame, *, group_value: float, feature_cols: list[str]) -> pd.DataFrame:
    rows = []
    group_df = wide.loc[pd.to_numeric(wide["study_group"], errors="coerce").eq(group_value)].copy()
    for _, row in group_df.iterrows():
        subject_id = str(row["ID"]).replace("TRACKFA_", "")
        for visit in (1, 2, 3):
            out = {
                "subject_id": subject_id,
                "visit": visit,
                "time_years": float(visit - 1),
                "study_group": group_value,
            }
            for feature in feature_cols:
                out[feature] = row.get(f"{feature}_v{visit}", np.nan)
            rows.append(out)
    return pd.DataFrame(rows)


def complete_interval_count(long_df: pd.DataFrame, *, start_visit: int, end_visit: int, feature_cols: list[str]) -> int:
    sub = long_df.loc[long_df["visit"].isin([start_visit, end_visit]), ["subject_id", "visit", *feature_cols]].dropna()
    counts = sub.groupby("subject_id")["visit"].nunique()
    return int((counts == 2).sum())

frda_wide_long = wide_to_visit_long(wide_df, group_value=0.0, feature_cols=imaging_cols)
control_long = wide_to_visit_long(wide_df, group_value=1.0, feature_cols=imaging_cols)

availability = pd.DataFrame([
    {"group": "FRDA", "subjects": frda_wide_long["subject_id"].nunique(), "V1->V2 complete": complete_interval_count(frda_wide_long, start_visit=1, end_visit=2, feature_cols=imaging_cols), "V2->V3 complete": complete_interval_count(frda_wide_long, start_visit=2, end_visit=3, feature_cols=imaging_cols), "V1->V3 complete": complete_interval_count(frda_wide_long, start_visit=1, end_visit=3, feature_cols=imaging_cols)},
    {"group": "Control", "subjects": control_long["subject_id"].nunique(), "V1->V2 complete": complete_interval_count(control_long, start_visit=1, end_visit=2, feature_cols=imaging_cols), "V2->V3 complete": complete_interval_count(control_long, start_visit=2, end_visit=3, feature_cols=imaging_cols), "V1->V3 complete": complete_interval_count(control_long, start_visit=1, end_visit=3, feature_cols=imaging_cols)},
])
print("Complete-case availability using the locked MRI feature panel")
display(availability)


Complete-case availability using the locked MRI feature panel


,group,subjects,V1->V2 complete,V2->V3 complete,V1->V3 complete
0,FRDA,174,108,100,101
1,Control,95,58,59,59


## Fit Locked SRM On FRDA Annual Pairs, Then Score Controls

The fitted direction is learned from FRDA annual change vectors. The scaler is also fitted from FRDA rows only. No control data are used in any learned parameter.


In [3]:
frda_train_cols = ["pair_id", "subject", "visit", *imaging_cols]
frda_train = pair_long_df[frda_train_cols].dropna().copy()
frda_train["visit"] = pd.to_numeric(frda_train["visit"], errors="coerce").astype(int)

X_frda = frda_train[imaging_cols].to_numpy(dtype=float)
X_frda_s, _, frda_mu, frda_sd = standardize_train_test(X_frda, X_frda)
if LOCKED_MODEL["z_clip"] is not None:
    clip = float(LOCKED_MODEL["z_clip"])
    X_frda_s = np.clip(X_frda_s, -clip, clip)

locked_srm = SRMGlobalLinear(
    ridge=LOCKED_MODEL["ridge"],
    covariance_shrinkage=LOCKED_MODEL["covariance_shrinkage"],
    start_visit=1,
    end_visit=2,
).fit(
    X_frda_s,
    frda_train["pair_id"].to_numpy(),
    frda_train["visit"].to_numpy(),
)

control_score_input = control_long[["subject_id", "visit", "time_years", *imaging_cols]].dropna().copy()
X_control = control_score_input[imaging_cols].to_numpy(dtype=float)
X_control_s = (X_control - frda_mu) / frda_sd
if LOCKED_MODEL["z_clip"] is not None:
    X_control_s = np.clip(X_control_s, -clip, clip)
control_score_input["score"] = locked_srm.score(X_control_s)

fit_summary = pd.DataFrame([
    {"step": "FRDA training rows after complete-case feature filter", "value": frda_train.shape[0]},
    {"step": "FRDA annual pairs used for fitting", "value": frda_train["pair_id"].nunique()},
    {"step": "FRDA participants represented in fitting", "value": frda_train["subject"].nunique()},
    {"step": "Control visit rows scored", "value": control_score_input.shape[0]},
    {"step": "Control participants scored", "value": control_score_input["subject_id"].nunique()},
    {"step": "MRI features in locked model", "value": len(imaging_cols)},
])
print("Locked model fit / score summary")
display(fit_summary)


Locked model fit / score summary


,step,value
0,FRDA training rows after complete-case feature...,414
1,FRDA annual pairs used for fitting,207
2,FRDA participants represented in fitting,117
3,Control visit rows scored,199
4,Control participants scored,77
5,MRI features in locked model,146


## Control Cohen's d_z Specificity Results

For controls, values near zero are desirable because healthy/control participants should not show strong disease-direction progression. Positive values mean the FRDA-trained biomarker increased in the FRDA progression direction even in controls; negative values mean the score moved opposite to that direction.


In [4]:
control_interval_rows = []
for start_visit, end_visit, label, annualise in [
    (1, 2, "V1->V2", True),
    (2, 3, "V2->V3", True),
    (1, 3, "V1->V3", True),
]:
    part = interval_effect_summary(
        control_score_input,
        subject_col="subject_id",
        visit_col="visit",
        score_col="score",
        time_col="time_years",
        intervals=[(start_visit, end_visit, label, annualise)],
        n_boot=N_BOOT,
        seed=RANDOM_SEED,
    )
    control_interval_rows.append(part)

control_interval_summary = pd.concat(control_interval_rows, ignore_index=True)
control_interval_summary.insert(0, "group", "Control")
control_interval_summary["model"] = LOCKED_MODEL["model"]

# Pooled annual specificity uses one row per annual control interval while retaining subject IDs for audit.
control_pairs_for_pool = []
for start_visit, end_visit, suffix in [(1, 2, "V1V2"), (2, 3, "V2V3")]:
    interval_scores = control_score_input.loc[control_score_input["visit"].isin([start_visit, end_visit])].copy()
    complete_subjects = interval_scores.groupby("subject_id")["visit"].nunique()
    complete_subjects = complete_subjects[complete_subjects == 2].index
    interval_scores = interval_scores[interval_scores["subject_id"].isin(complete_subjects)].copy()
    interval_scores["pair_id"] = interval_scores["subject_id"].astype(str) + "_" + suffix
    interval_scores["annual_interval"] = suffix
    interval_scores["visit"] = interval_scores["visit"].map({start_visit: 1, end_visit: 2})
    control_pairs_for_pool.append(interval_scores)
control_pair_score_long = pd.concat(control_pairs_for_pool, ignore_index=True)
control_pooled_annual = pooled_adjacent_pair_effect_summary(
    control_pair_score_long,
    pair_col="pair_id",
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
control_pooled_annual.insert(0, "group", "Control")
control_pooled_annual["model"] = LOCKED_MODEL["model"]

control_specificity_results = pd.concat([control_interval_summary, control_pooled_annual], ignore_index=True)
control_specificity_results = control_specificity_results.rename(columns={
    "mean_change": "mean_delta",
    "sd_change": "sd_delta",
    "d_z_ci_low": "ci_low",
    "d_z_ci_high": "ci_high",
    "p_delta_positive": "p_delta_gt_0",
})
control_specificity_results = control_specificity_results[[
    "group", "model", "interval", "n_pairs", "mean_delta", "sd_delta", "d_z", "ci_low", "ci_high", "p_delta_gt_0"
]]
print("Control specificity Cohen's d_z values")
display(control_specificity_results)

out_path = REPO_ROOT / "results" / "control_specificity_results.csv"
control_specificity_results.to_csv(out_path, index=False)
print(f"Saved control specificity results: {out_path}")


Control specificity Cohen's d_z values


,group,model,interval,n_pairs,mean_delta,sd_delta,d_z,ci_low,ci_high,p_delta_gt_0
0,Control,SRM Global Linear Composite,V1->V2,58,0.365142,1.468501,0.248649,-0.003059,0.525353,0.568966
1,Control,SRM Global Linear Composite,V2->V3,59,0.633232,1.741526,0.363608,0.118744,0.688394,0.644068
2,Control,SRM Global Linear Composite,V1->V3,59,0.559529,0.903701,0.619153,0.398068,0.887317,0.762712
3,Control,SRM Global Linear Composite,V1->V2 + V2->V3,117,0.500333,1.610663,0.310638,0.126253,0.511027,0.606838


Saved control specificity results: /Users/robertwang/Documents/New_project/biomarkers/results/control_specificity_results.csv


## Compare Controls Against Current FRDA OOF Results

This table uses the current model-performance notebook output for the FRDA OOF reference, then places the new control specificity result beside it.


In [5]:
def parse_first_float(value) -> float:
    text = str(value)
    try:
        return float(text.split()[0])
    except Exception:
        return np.nan

srm_log = pd.read_csv(SRM_LOG_PATH) if SRM_LOG_PATH.exists() else pd.DataFrame()
reference_rows = []
if not srm_log.empty:
    srm_log["mean_validation_annual_dz"] = pd.to_numeric(srm_log["mean_validation_annual_dz"], errors="coerce")
    srm_log["annual_interval_gap"] = pd.to_numeric(srm_log["annual_interval_gap"], errors="coerce")
    srm_best = (
        srm_log.dropna(subset=["mean_validation_annual_dz"])
        .sort_values(["mean_validation_annual_dz", "annual_interval_gap"], ascending=[False, True])
        .iloc[0]
    )
    reference_rows.extend([
        {"interval": "V1->V2", "frda_oof_d_z": srm_best.get("dz_v1_v2", np.nan), "frda_n": 108},
        {"interval": "V2->V3", "frda_oof_d_z": srm_best.get("dz_v2_v3", np.nan), "frda_n": 99},
        {"interval": "V1->V2 + V2->V3", "frda_oof_d_z": srm_best.get("d_score", np.nan), "frda_n": 207},
    ])

performance_df = pd.read_csv(MODEL_PERFORMANCE_PATH) if MODEL_PERFORMANCE_PATH.exists() else pd.DataFrame()
if not performance_df.empty:
    v13 = performance_df.loc[
        performance_df["model"].astype(str).eq("SRM Global Linear")
        & performance_df["question"].astype(str).eq("24-month cumulative sensitivity")
    ].head(1)
    if not v13.empty:
        reference_rows.append({
            "interval": "V1->V3",
            "frda_oof_d_z": parse_first_float(v13.iloc[0].get("value", np.nan)),
            "frda_n": v13.iloc[0].get("n", 90),
        })
frda_reference = pd.DataFrame(reference_rows)

comparison = control_specificity_results.rename(columns={
    "d_z": "control_d_z",
    "n_pairs": "control_n",
    "p_delta_gt_0": "control_p_delta_gt_0",
}).merge(frda_reference, on="interval", how="left")
comparison = comparison[["interval", "frda_oof_d_z", "frda_n", "control_d_z", "control_n", "control_p_delta_gt_0", "ci_low", "ci_high"]]
comparison_with_reference = comparison.dropna(subset=["frda_oof_d_z"]).reset_index(drop=True)
comparison_without_reference = comparison.loc[comparison["frda_oof_d_z"].isna(), ["interval", "control_d_z", "control_n", "control_p_delta_gt_0", "ci_low", "ci_high"]].reset_index(drop=True)

print("FRDA OOF reference from SRM optimization log vs control specificity")
display(comparison_with_reference)
if not comparison_without_reference.empty:
    print("Control intervals without a stored FRDA OOF reference in the annual-pair SRM log")
    display(comparison_without_reference)

print("Interpretation summary")
for _, row in comparison_with_reference.iterrows():
    print(
        f"{row['interval']}: control d_z={row['control_d_z']:.3f} "
        f"(N={int(row['control_n'])}, P(delta>0)={row['control_p_delta_gt_0']:.3f}); "
        f"FRDA OOF reference d_z={row['frda_oof_d_z']:.3f}."
    )


FRDA OOF reference from SRM optimization log vs control specificity


,interval,frda_oof_d_z,frda_n,control_d_z,control_n,control_p_delta_gt_0,ci_low,ci_high
0,V1->V2,1.129187,108.0,0.248649,58,0.568966,-0.003059,0.525353
1,V2->V3,0.779868,99.0,0.363608,59,0.644068,0.118744,0.688394
2,V1->V3,1.426891,90.0,0.619153,59,0.762712,0.398068,0.887317
3,V1->V2 + V2->V3,0.935054,207.0,0.310638,117,0.606838,0.126253,0.511027


Interpretation summary
V1->V2: control d_z=0.249 (N=58, P(delta>0)=0.569); FRDA OOF reference d_z=1.129.
V2->V3: control d_z=0.364 (N=59, P(delta>0)=0.644); FRDA OOF reference d_z=0.780.
V1->V3: control d_z=0.619 (N=59, P(delta>0)=0.763); FRDA OOF reference d_z=1.427.
V1->V2 + V2->V3: control d_z=0.311 (N=117, P(delta>0)=0.607); FRDA OOF reference d_z=0.935.
